# 🏥 Medical Diagnosis & Research Assistant

This notebook replicates the **ClinisightAI** project in Colab, maintaining the same module structure as the original app (minus the MCP server).

### Pipeline Overview
1. **Symptom Extraction** — parse natural language for known symptoms
2. **Diagnosis** — use Groq LLM (`llama-3.3-70b-versatile`) to generate a diagnosis & cure suggestions
3. **PubMed Research** — fetch real medical articles from NCBI PubMed API
4. **Summarization** — summarize the research abstracts with the LLM
5. **Full Pipeline Demo** — run all steps end-to-end
6. **Save & Export** — write all `.py` files and zip the output

> **Setup:** Make sure `GROQ_API_KEY` is saved in Colab Secrets (🔑 icon in the left sidebar) before running.

**bold text**## Install Dependencies

In [1]:
!pip install -q groq langchain langchain-groq langchain-core beautifulsoup4 lxml requests fastapi uvicorn python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00


##  Load GROQ API Key from Colab Secrets

In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("✅ GROQ_API_KEY loaded successfully.")

✅ GROQ_API_KEY loaded successfully.


##  Create Project Directory Structure

In [3]:
import os

# Replicate original project structure
os.makedirs("Medical_Diagnosis_App/functions", exist_ok=True)

# Create __init__.py to make functions a proper package
with open("Medical_Diagnosis_App/functions/__init__.py", "w") as f:
    f.write("")

print("✅ Project directory structure created:")
for root, dirs, files in os.walk("Medical_Diagnosis_App"):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace("Medical_Diagnosis_App", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

✅ Project directory structure created:
Medical_Diagnosis_App/
  functions/
    __init__.py


##  Write `functions/symptom_extractor.py`

In [4]:
symptom_extractor_code = '''
import re

# Extended symptom vocabulary for richer extraction
SYMPTOM_PATTERNS = [
    r"headache", r"fever", r"nausea", r"fatigue", r"pain",
    r"cough", r"chills", r"vomiting", r"diarrhea", r"dizziness",
    r"shortness of breath", r"sore throat", r"runny nose",
    r"chest pain", r"back pain", r"muscle ache", r"joint pain",
    r"rash", r"swelling", r"insomnia", r"anxiety", r"depression",
    r"loss of appetite", r"weight loss", r"weight gain",
    r"palpitations", r"blurred vision", r"tinnitus",
    r"numbness", r"tingling", r"weakness"
]

def extract_symptoms(text: str) -> list:
    """
    Parse natural language text and return a deduplicated list of
    recognised medical symptoms.

    Args:
        text: Free-text symptom description from the patient.

    Returns:
        List of unique symptom strings found in the text.
    """
    text_lower = text.lower()
    found = set()
    for pattern in SYMPTOM_PATTERNS:
        if re.search(r"\\b" + pattern + r"\\b", text_lower):
            found.add(pattern)
    return sorted(found)
'''

with open("Medical_Diagnosis_App/functions/symptom_extractor.py", "w") as f:
    f.write(symptom_extractor_code.strip())

print("✅ symptom_extractor.py written")
print(open("Medical_Diagnosis_App/functions/symptom_extractor.py").read())

✅ symptom_extractor.py written
import re

# Extended symptom vocabulary for richer extraction
SYMPTOM_PATTERNS = [
    r"headache", r"fever", r"nausea", r"fatigue", r"pain",
    r"cough", r"chills", r"vomiting", r"diarrhea", r"dizziness",
    r"shortness of breath", r"sore throat", r"runny nose",
    r"chest pain", r"back pain", r"muscle ache", r"joint pain",
    r"rash", r"swelling", r"insomnia", r"anxiety", r"depression",
    r"loss of appetite", r"weight loss", r"weight gain",
    r"palpitations", r"blurred vision", r"tinnitus",
    r"numbness", r"tingling", r"weakness"
]

def extract_symptoms(text: str) -> list:
    """
    Parse natural language text and return a deduplicated list of
    recognised medical symptoms.

    Args:
        text: Free-text symptom description from the patient.

    Returns:
        List of unique symptom strings found in the text.
    """
    text_lower = text.lower()
    found = set()
    for pattern in SYMPTOM_PATTERNS:
        if re.search(r"\b" + pa

##  Write `functions/diagnosis_symptoms.py`

In [5]:
diagnosis_symptoms_code = '''
import os
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

# Initialise Groq client via LangChain (>=1.2 compatible)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=1024,
    api_key=os.environ["GROQ_API_KEY"]
)

def get_diagnosis(symptoms: list) -> str:
    """
    Given a list of symptom strings, query the LLM for a possible
    diagnosis and suggested cure.

    Args:
        symptoms: List of symptom strings (e.g. [\\"fever\\", \\"cough\\"]).

    Returns:
        LLM-generated diagnosis and cure suggestion as a string.
    """
    if not symptoms:
        return "No symptoms detected. Please provide a symptom description."

    prompt = (
        f"Patient has symptoms: {\', \'.join(symptoms)}. "
        "Suggest possible medical diagnosis and a possible cure for the same. "
        "Be concise and structured."
    )

    messages = [
        SystemMessage(content="You are a helpful medical assistant. Provide structured, clear responses."),
        HumanMessage(content=prompt)
    ]

    response = llm.invoke(messages)
    return response.content.strip()
'''

with open("Medical_Diagnosis_App/functions/diagnosis_symptoms.py", "w") as f:
    f.write(diagnosis_symptoms_code.strip())

print("✅ diagnosis_symptoms.py written")
print(open("Medical_Diagnosis_App/functions/diagnosis_symptoms.py").read())

✅ diagnosis_symptoms.py written
import os
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

# Initialise Groq client via LangChain (>=1.2 compatible)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=1024,
    api_key=os.environ["GROQ_API_KEY"]
)

def get_diagnosis(symptoms: list) -> str:
    """
    Given a list of symptom strings, query the LLM for a possible
    diagnosis and suggested cure.

    Args:
        symptoms: List of symptom strings (e.g. [\"fever\", \"cough\"]).

    Returns:
        LLM-generated diagnosis and cure suggestion as a string.
    """
    if not symptoms:
        return "No symptoms detected. Please provide a symptom description."

    prompt = (
        f"Patient has symptoms: {', '.join(symptoms)}. "
        "Suggest possible medical diagnosis and a possible cure for the same. "
        "Be concise and structured."
    )

    messages = [
        SystemMessage(content

##  — Write `functions/pubmed_articles.py`

In [25]:
pubmed_articles_code = '''
import requests
from bs4 import BeautifulSoup

def fetch_pubmed_articles_with_metadata(query: str, max_results: int = 3, use_mock_if_empty: bool = True) -> list:
    """
    Search PubMed via the NCBI E-utilities API and return structured
    metadata for the top articles matching the query.

    Args:
        query:            Search term (space-joined symptoms work well).
        max_results:      Maximum number of articles to return.
        use_mock_if_empty: Return mock data if no real results are found.

    Returns:
        List of dicts with keys: title, abstract, authors,
        publication_date, article_url.
    """
    headers = {"User-Agent": "Mozilla/5.0 (ClinisightAI-Research-Tool)"}

    # Step 1: Search PubMed for article IDs
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    try:
        search_response = requests.get(
            search_url, params=search_params, headers=headers, timeout=10
        ).json()
        id_list = search_response["esearchresult"]["idlist"]
        print("Found PubMed IDs:", id_list)

        if not id_list:
            raise ValueError("No IDs found for this query.")

        ids = ",".join(id_list)

        # Step 2: Fetch full XML records
        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        fetch_params = {"db": "pubmed", "id": ids, "retmode": "xml"}
        fetch_response = requests.get(
            fetch_url, params=fetch_params, headers=headers, timeout=10
        )
        soup = BeautifulSoup(fetch_response.text, "lxml")
        articles_xml = soup.find_all("pubmedarticle")
        print("Articles found in XML:", len(articles_xml))

        articles_info = []
        for article, pmid in zip(articles_xml, id_list):
            title_tag    = article.find("articletitle")
            abstract_tag = article.find("abstract")
            date_tag     = article.find("pubdate")
            author_tags  = article.find_all("author")

            title    = title_tag.get_text(strip=True) if title_tag else "No title"
            abstract = abstract_tag.get_text(separator=" ", strip=True) if abstract_tag else "No abstract available"

            authors = []
            for author in author_tags:
                last = author.find("lastname")
                fore = author.find("forename")
                if last and fore:
                    authors.append(f"{fore.get_text()} {last.get_text()}")
                elif last:
                    authors.append(last.get_text())
            authors = authors if authors else ["No authors listed"]

            pub_date = "No date"
            if date_tag:
                year  = date_tag.find("year")
                month = date_tag.find("month")
                if year and month:
                    pub_date = f"{month.get_text()} {year.get_text()}"
                elif year:
                    pub_date = year.get_text()

            url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
            print(f"""Article: {title}
   - Authors: {authors}
   - Date: {pub_date}
   - URL: {url}
""")

            articles_info.append({
                "title": title,
                "abstract": abstract,
                "authors": authors,
                "publication_date": pub_date,
                "article_url": url
            })

        if not articles_info and use_mock_if_empty:
            print("No valid articles found, returning mock data.")
            return _mock_data()

        return articles_info

    except Exception as e:
        print(f"Error during PubMed fetch: {e}")
        return _mock_data() if use_mock_if_empty else [{"message": f"Error: {e}"}]


def _mock_data() -> list:
    """Return a fallback mock article when PubMed is unreachable."""
    return [{
        "title": "Simulated Study on Fever",
        "abstract": "This is a simulated abstract on the treatment of fever in adults.",
        "authors": ["John Doe", "Jane Smith"],
        "publication_date": "March 2024",
        "article_url": "https://pubmed.ncbi.nlm.nih.gov/12345678/"
    }]
'''

with open("Medical_Diagnosis_App/functions/pubmed_articles.py", "w") as f:
    f.write(pubmed_articles_code.strip())

print("✅ pubmed_articles.py written")
print(open("Medical_Diagnosis_App/functions/pubmed_articles.py").read())

✅ pubmed_articles.py written
import requests
from bs4 import BeautifulSoup

def fetch_pubmed_articles_with_metadata(query: str, max_results: int = 3, use_mock_if_empty: bool = True) -> list:
    """
    Search PubMed via the NCBI E-utilities API and return structured
    metadata for the top articles matching the query.

    Args:
        query:            Search term (space-joined symptoms work well).
        max_results:      Maximum number of articles to return.
        use_mock_if_empty: Return mock data if no real results are found.

    Returns:
        List of dicts with keys: title, abstract, authors,
        publication_date, article_url.
    """
    headers = {"User-Agent": "Mozilla/5.0 (ClinisightAI-Research-Tool)"}

    # Step 1: Search PubMed for article IDs
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    try:

## — Write `functions/summerize_pubmed.py`

In [26]:
summerize_pubmed_code = '''
import os
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

# Groq LLM for summarisation (llama-instant is fast; versatile for quality)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=512,
    api_key=os.environ["GROQ_API_KEY"]
)

def summarize_text(text: str) -> str:
    """
    Summarise medical PubMed abstract text using Groq LLM.

    Args:
        text: Raw abstract text (concatenated from multiple articles).

    Returns:
        Concise summary string.
    """
    if not text or not text.strip():
        return "No abstract text provided to summarise."

    prompt = f"""Summarize the following medical research abstract in a clear, concise paragraph:

{text}"""

    messages = [
        SystemMessage(content="You are a medical research summarizer. Provide accurate, concise summaries."),
        HumanMessage(content=prompt)
    ]

    response = llm.invoke(messages)
    return response.content.strip()
'''

with open("Medical_Diagnosis_App/functions/summerize_pubmed.py", "w") as f:
    f.write(summerize_pubmed_code.strip())

print("✅ summerize_pubmed.py written")
print(open("Medical_Diagnosis_App/functions/summerize_pubmed.py").read())

✅ summerize_pubmed.py written
import os
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

# Groq LLM for summarisation (llama-instant is fast; versatile for quality)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=512,
    api_key=os.environ["GROQ_API_KEY"]
)

def summarize_text(text: str) -> str:
    """
    Summarise medical PubMed abstract text using Groq LLM.

    Args:
        text: Raw abstract text (concatenated from multiple articles).

    Returns:
        Concise summary string.
    """
    if not text or not text.strip():
        return "No abstract text provided to summarise."

    prompt = f"""Summarize the following medical research abstract in a clear, concise paragraph:

{text}"""

    messages = [
        SystemMessage(content="You are a medical research summarizer. Provide accurate, concise summaries."),
        HumanMessage(content=prompt)
    ]

    response = llm.invoke(message

##  — Write `app.py` (FastAPI App)

In [8]:
app_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
from functions.symptom_extractor import extract_symptoms
from functions.diagnosis_symptoms import get_diagnosis
from functions.pubmed_articles import fetch_pubmed_articles_with_metadata
from functions.summerize_pubmed import summarize_text

app = FastAPI(title="ClinisightAI - Medical Diagnosis API")


class SymptomInput(BaseModel):
    description: str


@app.post("/diagnosis")
def diagnosis(data: SymptomInput):
    """
    Full medical diagnosis pipeline:
    1. Extract symptoms from free-text description.
    2. Generate LLM diagnosis.
    3. Fetch PubMed research articles.
    4. Summarise research abstracts.
    """
    symptom = extract_symptoms(data.description)
    diagnosis_result = get_diagnosis(symptom)
    pubmed_articles = fetch_pubmed_articles_with_metadata(" ".join(symptom))

    # Concatenate abstracts (cap at 3000 chars to stay within token limits)
    combined_abstracts = " ".join(
        a.get("abstract", "") for a in pubmed_articles
    )[:3000]
    summary = summarize_text(combined_abstracts)

    return {
        "symptom": symptom,
        "diagnosis": diagnosis_result,
        "pubmed_articles": pubmed_articles,
        "pubmed_summary": summary
    }


if __name__ == "__main__":
    import uvicorn
    uvicorn.run("app:app", host="0.0.0.0", port=8080, reload=True)
'''

with open("Medical_Diagnosis_App/app.py", "w") as f:
    f.write(app_code.strip())

print("✅ app.py written")
print(open("Medical_Diagnosis_App/app.py").read())

✅ app.py written
from fastapi import FastAPI
from pydantic import BaseModel
from functions.symptom_extractor import extract_symptoms
from functions.diagnosis_symptoms import get_diagnosis
from functions.pubmed_articles import fetch_pubmed_articles_with_metadata
from functions.summerize_pubmed import summarize_text

app = FastAPI(title="ClinisightAI - Medical Diagnosis API")


class SymptomInput(BaseModel):
    description: str


@app.post("/diagnosis")
def diagnosis(data: SymptomInput):
    """
    Full medical diagnosis pipeline:
    1. Extract symptoms from free-text description.
    2. Generate LLM diagnosis.
    3. Fetch PubMed research articles.
    4. Summarise research abstracts.
    """
    symptom = extract_symptoms(data.description)
    diagnosis_result = get_diagnosis(symptom)
    pubmed_articles = fetch_pubmed_articles_with_metadata(" ".join(symptom))

    # Concatenate abstracts (cap at 3000 chars to stay within token limits)
    combined_abstracts = " ".join(
        a.ge

##  `main.py`

In [9]:
main_code = '''
def main():
    print("Hello from ClinisightAI!")


if __name__ == "__main__":
    main()
'''

with open("Medical_Diagnosis_App/main.py", "w") as f:
    f.write(main_code.strip())

print("✅ main.py written")
print(open("Medical_Diagnosis_App/main.py").read())

✅ main.py written
def main():
    print("Hello from ClinisightAI!")


if __name__ == "__main__":
    main()


##  `requirements.txt`

In [10]:
requirements = """groq
langchain>=0.2.0
langchain-groq>=0.1.0
langchain-core>=0.2.0
python-dotenv
beautifulsoup4
lxml
requests
fastapi
uvicorn
"""

with open("Medical_Diagnosis_App/requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt written")
print(open("Medical_Diagnosis_App/requirements.txt").read())

✅ requirements.txt written
groq
langchain>=0.2.0
langchain-groq>=0.1.0
langchain-core>=0.2.0
python-dotenv
beautifulsoup4
lxml
requests
fastapi
uvicorn



##  Add Project Root to Python Path

In [11]:
import sys
import os

# So that 'from functions.xxx import yyy' works from the notebook
project_root = os.path.abspath("Medical_Diagnosis_App")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("✅ Project root added to sys.path:", project_root)

✅ Project root added to sys.path: /content/Medical_Diagnosis_App


---
## 🧪 Testing Each Module Independently

### Test: Symptom Extractor

In [12]:
from functions.symptom_extractor import extract_symptoms

test_description = "I have been having a bad headache, high fever, and I feel nauseous and very fatigued."
symptoms = extract_symptoms(test_description)

print("Input:", test_description)
print("Extracted symptoms:", symptoms)

Input: I have been having a bad headache, high fever, and I feel nauseous and very fatigued.
Extracted symptoms: ['fever', 'headache']


### Test: Diagnosis (Groq LLM)

In [13]:
from functions.diagnosis_symptoms import get_diagnosis

diagnosis_result = get_diagnosis(symptoms)

print("Symptoms passed:", symptoms)
print("\n--- Diagnosis & Cure Suggestion ---")
print(diagnosis_result)

Symptoms passed: ['fever', 'headache']

--- Diagnosis & Cure Suggestion ---
**Possible Medical Diagnosis:**
1. Viral Infection (e.g., Flu, Common Cold)
2. Bacterial Infection (e.g., Sinusitis, Meningitis)
3. Other conditions (e.g., Malaria, Typhoid)

**Possible Cure:**
1. **Medication:** 
   - Antipyretics (e.g., Paracetamol, Ibuprofen) for fever reduction
   - Antibiotics (if bacterial infection is confirmed)
2. **Rest and Hydration:** Adequate rest and fluid intake to help the body recover
3. **Further Evaluation:** Consult a doctor for a thorough diagnosis and treatment plan.


from functions.pubmed_articles import fetch_pubmed_articles_with_metadata

query = " ".join(symptoms)
print(f"Searching PubMed for: '{query}'\n")

articles = fetch_pubmed_articles_with_metadata(query, max_results=3)

print("\n--- PubMed Articles ---")
for i, article in enumerate(articles, 1):
    print(f"\n[{i}] {article['title']}")
    print(f"    Authors: {', '.join(article['authors'][:3])}")
    print(f"    Date: {article['publication_date']}")
    print(f"    URL: {article['article_url']}")
    print(f"    Abstract snippet: {article['abstract'][:200]}...")

In [27]:
from functions.pubmed_articles import fetch_pubmed_articles_with_metadata

query = " ".join(symptoms)
print(f"Searching PubMed for: '{query}'\n")

articles = fetch_pubmed_articles_with_metadata(query, max_results=3)

print("\n--- PubMed Articles ---")
for i, article in enumerate(articles, 1):
    print(f"\n[{i}] {article['title']}")
    print(f"    Authors: {', '.join(article['authors'][:3])}")
    print(f"    Date: {article['publication_date']}")
    print(f"    URL: {article['article_url']}")
    print(f"    Abstract snippet: {article['abstract'][:200]}...")

Searching PubMed for: 'fever headache'

Found PubMed IDs: ['42086839', '42083698', '42078631']
Articles found in XML: 3
Article: Clinical features and therapeutic outcomes in pediatric patients with Castleman's disease: a retrospective cohort analysis.
   - Authors: ['Yuanyuan Wang', 'Zixin Qin', 'Shu Xin Chen', 'Ru Ting Shi', 'Jia Ning Liu', 'Yongcheng Fu', 'Jingyue Wang', 'Shangkun Li', 'Tan Xie', 'Da Zhang']
   - Date: May 2026
   - URL: https://pubmed.ncbi.nlm.nih.gov/42086839/

Article: Rupatadine as an Add-On Therapy for Dengue Hemorrhagic Fever: A Case Report.
   - Authors: ['Mauricio E Flores']
   - Date: Apr 2026
   - URL: https://pubmed.ncbi.nlm.nih.gov/42083698/

Article: Pulmonary valve endocarditis in a postpartum patient: a case report of unconventional risk.
   - Authors: ['Bishweshwar Joshi', 'Dhiraj Adhikari', 'Bishal Budha', 'Neetika Paudel', 'Shivam Jha', 'Tekraj Upadhaya']
   - Date: May 2026
   - URL: https://pubmed.ncbi.nlm.nih.gov/42078631/


--- PubMed Articles 

/content/Medical_Diagnosis_App/functions/pubmed_articles.py:46: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(fetch_response.text, "lxml")


### Test: PubMed Summarizer (Groq LLM)

In [28]:
from functions.summerize_pubmed import summarize_text

combined_abstracts = " ".join(
    a.get("abstract", "") for a in articles
)[:3000]

summary = summarize_text(combined_abstracts)

print("--- Research Summary ---")
print(summary)

--- Research Summary ---
Here's a clear and concise summary of the medical research abstract:

This study examined 30 pediatric patients with Castleman disease (CD), a rare lymphoproliferative disorder. The patients were divided into two groups: unicentric CD (UCD) with 19 patients and multicentric CD (MCD) with 11 patients. The study found that UCD patients were younger at diagnosis, had a shorter diagnostic delay, and were more likely to have higher levels of certain blood cells and proteins. In contrast, MCD patients were older at diagnosis and had higher levels of platelets and globulin. The study also identified differences in treatment approaches, with UCD patients more likely to undergo surgery and MCD patients more likely to receive chemotherapy. Additionally, the study found that certain laboratory parameters, such as mean corpuscular volume (MCV) and hemoglobin (Hb), were associated with outcomes in CD patients. The study also compared single-center and multicenter CD cohorts

---
## 🚀 Full Pipeline — End-to-End Demo

In [29]:
import json
from functions.symptom_extractor import extract_symptoms
from functions.diagnosis_symptoms import get_diagnosis
from functions.pubmed_articles import fetch_pubmed_articles_with_metadata
from functions.summerize_pubmed import summarize_text

def run_diagnosis_pipeline(patient_description: str) -> dict:
    """
    Full ClinisightAI pipeline:
    symptom extraction → LLM diagnosis → PubMed research → summarization.

    Args:
        patient_description: Free-text patient symptom input.

    Returns:
        Dict with symptoms, diagnosis, pubmed_articles, pubmed_summary.
    """
    print("="*60)
    print("ClinisightAI — Medical Diagnosis & Research Assistant")
    print("="*60)

    # Step 1: Extract symptoms
    print("\n[Step 1] Extracting symptoms...")
    symptom = extract_symptoms(patient_description)
    print(f"  → Found: {symptom}")

    # Step 2: Get diagnosis
    print("\n[Step 2] Generating diagnosis via Groq LLM...")
    diagnosis_result = get_diagnosis(symptom)
    print("  → Diagnosis generated.")

    # Step 3: Fetch PubMed articles
    print("\n[Step 3] Fetching PubMed research articles...")
    pubmed_articles = fetch_pubmed_articles_with_metadata(
        " ".join(symptom), max_results=3
    )
    print(f"  → Retrieved {len(pubmed_articles)} article(s).")

    # Step 4: Summarise abstracts
    print("\n[Step 4] Summarising research abstracts...")
    combined_abstracts = " ".join(
        a.get("abstract", "") for a in pubmed_articles
    )[:3000]
    pubmed_summary = summarize_text(combined_abstracts)
    print("  → Summary generated.")

    result = {
        "symptom": symptom,
        "diagnosis": diagnosis_result,
        "pubmed_articles": pubmed_articles,
        "pubmed_summary": pubmed_summary
    }

    print("\n" + "="*60)
    print("RESULT")
    print("="*60)
    print(f"\nSymptoms Detected : {result['symptom']}")
    print(f"\nDiagnosis:\n{result['diagnosis']}")
    print(f"\nPubMed Research Summary:\n{result['pubmed_summary']}")
    print(f"\nPubMed Articles ({len(result['pubmed_articles'])}):")
    for i, a in enumerate(result['pubmed_articles'], 1):
        print(f"  [{i}] {a['title']} — {a['article_url']}")

    return result


# --- Run the pipeline ---
patient_input = "I have been feeling a severe headache, fever with chills, nausea, and extreme fatigue for the past two days."
result = run_diagnosis_pipeline(patient_input)

ClinisightAI — Medical Diagnosis & Research Assistant

[Step 1] Extracting symptoms...
  → Found: ['chills', 'fatigue', 'fever', 'headache', 'nausea']

[Step 2] Generating diagnosis via Groq LLM...
  → Diagnosis generated.

[Step 3] Fetching PubMed research articles...
Found PubMed IDs: ['41248143', '41244967', '40981328']


/content/Medical_Diagnosis_App/functions/pubmed_articles.py:46: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(fetch_response.text, "lxml")


Articles found in XML: 3
Article: Active vaccine safety surveillance: Experience from a prospective cohort event monitoring study of COVID-19 vaccines in Kenya.
   - Authors: ['Don B Odhiambo', 'Donald Akech', 'Boniface Karia', 'Makobu Kimani', 'Samuel Sang', 'Antipa Sigilai', 'Shirine Voller', 'Christine Mataza', "David Mang'ong'o", "Rose Jalang'o", 'Martha Mandale', 'Anthony O Etyang', 'John Anthony Gerard Scott', 'Ambrose Agweyu', 'Eunice Wangeci Kagucia']
   - Date: 2025
   - URL: https://pubmed.ncbi.nlm.nih.gov/41248143/

Article: Bacteremia caused byHelicobacter trogontumindicative of zoonotic infection in a pig farmer: a case report.
   - Authors: ['Nobumasa Hojo', 'Takashi Unehara', 'Masato Suzuki', 'Michio Suzuki', 'Emiko Rimbara']
   - Date: Sep 2025
   - URL: https://pubmed.ncbi.nlm.nih.gov/41244967/

Article: Sex, Age, and COVID-19 Vaccine Characteristics Associated with Adverse Events After Vaccination and Severity: A Retrospective Analysis.
   - Authors: ['Edgar P Rodrígu

---
## 💾 Save Output JSON

In [30]:
import json

os.makedirs("Medical_Diagnosis_App/outputs", exist_ok=True)

output_path = "Medical_Diagnosis_App/outputs/diagnosis_result.json"
with open(output_path, "w") as f:
    json.dump(result, f, indent=2)

print(f"✅ Diagnosis result saved to: {output_path}")
print(json.dumps(result, indent=2)[:1000], "...")

✅ Diagnosis result saved to: Medical_Diagnosis_App/outputs/diagnosis_result.json
{
  "symptom": [
    "chills",
    "fatigue",
    "fever",
    "headache",
    "nausea"
  ],
  "diagnosis": "**Possible Medical Diagnosis:**\nBased on the symptoms, possible diagnoses include:\n1. Influenza (flu)\n2. Viral gastroenteritis (stomach flu)\n3. Urinary tract infection (UTI)\n4. Pneumonia\n5. Meningitis (less likely, but requires immediate attention)\n\n**Possible Cure/Treatment:**\nTreatment depends on the underlying cause, but general measures include:\n1. Rest and hydration\n2. Over-the-counter medications (e.g., acetaminophen or ibuprofen) for fever and headache\n3. Anti-nausea medications (if prescribed)\n4. Antibiotics (if bacterial infection is confirmed)\n5. Antiviral medications (if influenza is confirmed)\n\n**Next Steps:**\nIt is essential to consult a healthcare professional for a proper diagnosis and personalized treatment plan.",
  "pubmed_articles": [
    {
      "title": "Active 

---
## 📦 Zip All Output Files for Download

In [31]:
import zipfile
import os

zip_path = "/content/Medical_Diagnosis_App_output.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk("Medical_Diagnosis_App"):
        # Skip __pycache__ directories
        dirs[:] = [d for d in dirs if d != "__pycache__"]
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=".")
            zipf.write(file_path, arcname)
            print(f"  Added: {arcname}")

print(f"\n✅ All files zipped to: {zip_path}")
print(f"   Zip size: {os.path.getsize(zip_path) / 1024:.1f} KB")

  Added: Medical_Diagnosis_App/main.py
  Added: Medical_Diagnosis_App/requirements.txt
  Added: Medical_Diagnosis_App/app.py
  Added: Medical_Diagnosis_App/functions/diagnosis_symptoms.py
  Added: Medical_Diagnosis_App/functions/symptom_extractor.py
  Added: Medical_Diagnosis_App/functions/pubmed_articles.py
  Added: Medical_Diagnosis_App/functions/__init__.py
  Added: Medical_Diagnosis_App/functions/summerize_pubmed.py
  Added: Medical_Diagnosis_App/outputs/diagnosis_result.json

✅ All files zipped to: /content/Medical_Diagnosis_App_output.zip
   Zip size: 9.3 KB


##  Download the Zip

In [ ]:
from google.colab import files

files.download("/content/Medical_Diagnosis_App_output.zip")
print("✅ Download initiated for Medical_Diagnosis_App_output.zip")

---
## 📋 Project Structure Summary

```
Medical_Diagnosis_App/
├── app.py                        # FastAPI app (/diagnosis endpoint)
├── main.py                       # Entry point
├── requirements.txt              # Dependencies
├── functions/
│   ├── __init__.py
│   ├── symptom_extractor.py      # Regex-based symptom parser
│   ├── diagnosis_symptoms.py     # Groq LLM diagnosis (llama-3.3-70b-versatile)
│   ├── pubmed_articles.py        # NCBI PubMed E-utilities scraper
│   └── summerize_pubmed.py       # Groq LLM summarizer (llama-3.1-8b-instant)
└── outputs/
    └── diagnosis_result.json     # Saved pipeline output
```

### LLM Models Used (Groq)
| Task | Model | Reason |
|------|-------|--------|
| Diagnosis | `llama-3.3-70b-versatile` | High quality medical reasoning |
| Summarization | `llama-3.1-8b-instant` | Fast, efficient for summarization |

### Key Changes from Original (OpenAI → Groq)
- `openai.OpenAI` → `langchain_groq.ChatGroq` (LangChain ≥ 1.2)
- `client.chat.completions.create(...)` → `llm.invoke([SystemMessage, HumanMessage])`
- MCP server (`mcp_tool.py`) excluded as per requirements
- `OPENAI_API_KEY` → `GROQ_API_KEY` from Colab Secrets